# 05. 입금 기반 타당성 검증과 입금 경로 기술 (H-C1 · H-C2)
> 계획서: `계획/P5_위축기준과_입금검증.md` (Codex 1·2·4차 반영) · 코드: `code/validate_cnt.py`

**이 노트북이 하는 일** — 04에서 고정한 규칙(v2)으로 탐지한 기업이 **이후 6개월 입금이 상대적으로 더 약해졌는지**를 비교합니다. 입금은 탐지에 쓰지 않은 외부 기준입니다.
- **H-C1 (주 검정 1개)**: 층별 표준화 비율비 RR = 관찰 입금 약화 건수 ÷ 기대 건수(대조군 약화율 기준). 기업 단위 군집 부트스트랩, **양측 95% CI 하한 > 1 이면 채택**.
- **H-C2 (탐색)**: 탐지 전·중·후 입금 경로를 **기술**합니다. 선행·조기경보 판정은 하지 않습니다.
- **결론의 상한**: *"판정 가능하고 비교 가능한 기업에서 I1 최초 탐지가 향후 6개월의 상대적 입금 약화와 연관된다."* 인과나 '모든 규모'를 말하지 않습니다.

🔒 **분석 순서와 잠금**: 이 노트북은 먼저 **36개월 법인(P36)** 만 분석합니다. **35개월 법인(P35만·P35+)의 입금 결과는 `HOLDOUT_OPEN = False` 로 잠겨** 있고, P36 결과를 기록·해석한 뒤 안내에 따라서만 엽니다. P36에서 본 결과로 규칙을 바꾸지 않습니다. 바꾼다면 탐색으로 표시하고 잠긴 표본에서만 검증합니다.

**화면 출력은 집계값뿐** · 5 미만 칸은 `<5` · 법인ID 없음. 부트스트랩이 많아 몇 분 걸립니다 (`CNT_BOOT` 로 줄일 수 있음, 기본 2,000).

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "analysis" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 analysis/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

import detect_cnt as dc
import validate_cnt as vc

spec = json.load(open(WORK / "interim" / "detect_spec_v2.json", encoding="utf-8"))
rule, V1 = spec["rule"], spec["v1_rule"]
MODE, K, L, PHI2, N2 = rule["비교 기준"], int(rule["k"]), int(rule["L"]), float(rule["phi"]), float(rule["n_pct"])
B = int(os.environ.get("CNT_BOOT", "2000"))
# 잠금: 35개월 법인의 입금 결과는 P36 분석·기록이 끝난 뒤에만 연다 (안내 없이 True 로 바꾸지 마세요).
HOLDOUT_OPEN = True
print(f"고정 규칙(v2, 04에서 확정): 비교 기준 {MODE}, k={K}, Φ={PHI2:.2f}, n={N2:.0f}%   | v1(민감도): Φ={V1['phi']:.2f}, n={V1['n_pct']:.0f}%")
print("35개월 법인 입금 결과:", "열림" if HOLDOUT_OPEN else "🔒 잠김")

def sup(x):
    return cl.sup(int(round(x)))

def f_rr(r):
    return "정의 불가 (기대 약화 0건)" if not np.isfinite(r["rr"]) else f"{r['rr']:.2f} ({r['lo']:.2f}~{r['hi']:.2f})"

## 1. P36에서 고정하는 상수와 분석 준비
입금 약화 문턱 θ_O1(판정 가능 (기업, 기준월) 전체 입금 변화율의 5번째 백분위수), 규모 5분위 경계, 업종군 상위 5개, 입금 윈저화 상한 등은 **P36에서 한 번 정하고 이후 모든 표본에 그대로 적용**합니다.

In [ ]:
mk = masks["P36"]
C5 = vc.fit_constants(panel, mk, MODE, K, PHI2, N2)
D = vc.prepare(panel, mk, C5)
U, ER, F36 = D["units"], D["er"], D["F"]
ch, st = D["change"], D["status"]
assert int((D["det"] >= 0).sum()) == spec["power"]["P36"]["N1"], "04의 탐지 기업 수와 다릅니다 (04를 다시 실행하세요)"
levels_sec = panel.attr_levels["업종_대분류"]
SECN = [levels_sec[c] for c in C5["top_codes"]] + ["기타"]
tab = pd.DataFrame([
    ("입금 약화 문턱 θ_O1 (입금 변화율 g 의 5번째 백분위수)", round(C5["theta_o1"], 3)),
    ("잔액 약화 문턱 θ_O3", round(C5["theta_o3"], 3)),
    ("규모 5분위 경계 (비교 기간 평균 I1, 20·40·60·80%)", " / ".join(f"{x:.1f}" for x in C5["edges"])),
    ("업종군 (P36 기업 수 상위 5개 + 기타)", ", ".join(SECN)),
    ("입금 윈저화 상한 (P36 입금의 99.5 백분위수)", round(C5["cap_y"], 1)),
    ("탐지 전 입금 추세 문턱 θ_h (h_pre 5번째 백분위수, e ≥ 23)", round(C5["theta_h"], 3)),
    ("L_pre 3분위 경계 / h_pre 3분위 경계", " / ".join(f"{x:.1f}" for x in C5["l_terc"]) + "  |  " + " / ".join(f"{x:.3f}" for x in C5["h_terc"])),
], columns=["고정 상수", "값"])
cl.show_table(tab, "P36 에서 고정한 상수 (이후 모든 표본에 그대로 적용)", OUT / "nb05", "constants")
print(f"탐지 기업(04와 같은 수): {int((D['det'] >= 0).sum()):,}곳 → 결과 g 가 정의된 탐지 단위 {int(U.is_det.sum()):,} (정의 불가로 제외 {U.meta['탐지 중 결과 미정의로 제외']})")

## 2. ⑤-1 n의 안정성 확인 (보고만, n은 바꾸지 않음)
판정 기간을 2024년 월과 2025년 월로 나눈 변화율의 5번째 백분위수 차이가 **5%p 이하**이면 안정입니다. 차이가 커도 n은 35%로 고정하고 한계에 적습니다.

In [ ]:
def n0_of(es):
    v = np.concatenate([ch.r[:, e][st[:, e] == 0] for e in es])
    v = v[~np.isnan(v)]
    return -100 * float(np.percentile(v, 5)), int(v.size)
e24 = [e for e in ER if e <= 23]
e25 = [e for e in ER if e >= 24]
(a0, na), (b0, nb), (c0, nc) = n0_of(e24), n0_of(e25), n0_of(ER)
tab = pd.DataFrame([("2024년 기준월", len(e24), cl.sup(na), round(a0, 1)), ("2025년 기준월", len(e25), cl.sup(nb), round(b0, 1)), ("전체 (고정 n 의 근거)", len(ER), cl.sup(nc), round(c0, 1))],
                   columns=["구분", "기준월 수", "판정 가능 기업-월", "n₀ (%)"])
cl.show_table(tab, "n₀ = −100 × 5번째 백분위수 (판정 가능 기업-월의 변화율)", OUT / "nb05", "n_stability")
print(f"2024 vs 2025 차이 {abs(a0 - b0):.1f}%p → " + ("안정 (≤ 5%p)" if abs(a0 - b0) <= 5 else "불안정 (> 5%p): n 은 고정, 한계에 기록 (반분 n 은 P7 민감도)"))

## 3. ⑤-2 탐지군과 위험집합(비교군) 구성, 규모별 탐지
탐지군은 기업당 1건(최초 탐지월 t), 비교군은 각 t에 **판정 가능했고 t까지 한 번도 탐지되지 않은 기업**입니다(이후 탐지돼도 t월 비교군 자격을 유지). **규모별 문턱 통과율(기업-월)과 기업별 첫 탐지율은 서로 다른 양**이라 따로 보여 줍니다.

In [ ]:
Ud = U.is_det
print(f"탐지 기업 {int(Ud.sum()):,}곳 · 위험집합 기업-월 {int((~Ud).sum()):,}개 (고유 기업 {len(np.unique(U.firm[~Ud])):,}곳)")
rows = [{"기준월": int(panel.months[e]), "탐지 기업": cl.sup((U.e[Ud] == e).sum()), "위험집합 기업": cl.sup(((U.e == e) & ~Ud).sum())} for e in ER]
cl.show_table(pd.DataFrame(rows), "기준월별 탐지 기업과 위험집합 크기 (P36)", OUT / "nb05", "units_by_month")

jf = (st[:, ER] == 0)
size_m, r_m = D["size"][:, ER], ch.r[:, ER]
first = np.where(jf.any(1), jf.argmax(1), -1)
fsize = np.where(first >= 0, size_m[np.arange(len(first)), np.clip(first, 0, None)], -1)
det_any = D["det"] >= 0
rows = []
for q in range(5):
    cq = jf & (size_m == q)
    firms = fsize == q
    nd = int((det_any & firms).sum())
    rows.append({"규모 5분위": f"{q + 1}분위" + (" (최소)" if q == 0 else " (최대)" if q == 4 else ""), "판정 가능 기업": cl.sup(firms.sum()), "탐지 기업": cl.sup(nd),
                 "기업별 첫 탐지율 %": round(100 * nd / max(int(firms.sum()), 1), 1),
                 "기업-월 문턱 통과율 % (r ≤ −n)": round(100 * float((r_m[cq] <= -N2 / 100 + 1e-9).mean()), 1) if cq.any() else None})
SIZE_TAB = pd.DataFrame(rows)
cl.show_table(SIZE_TAB, "규모별 판정 가능·탐지: 기업-월 문턱 통과율과 기업별 첫 탐지율 (P36)", OUT / "nb05", "size_detect")

## 4. ⑤-3~⑤-5 H-C1 주 검정: 층별 표준화 비율비
RR = **관찰된 입금 약화 건수 ÷ 기대 건수**. 기대 건수 = 층(기준월 × 규모 5분위 × 업종군)별 탐지 기업 수 × 그 층 위험집합의 입금 약화율 (표준화 대상 = 탐지군의 층 분포). 대조군이 5개 미만인 층은 업종을 합치고, 그래도 모자라면 해당 탐지군을 **제외(외삽하지 않음)** 합니다. 불확실성은 **기업 이력 단위 군집 부트스트랩**(기업의 모든 기준월 기록을 함께 재표집, 매번 층·공통 지지·표준화를 다시 계산)입니다.

In [ ]:
t0 = time.time()
R = vc.analyze_units(U, F36, B, seed=20261001)
tab = pd.DataFrame([{
    "탐지 기업 N1": cl.sup(R["n1"]), "공통 지지 안의 탐지 기업": cl.sup(R["n1_supported"]), "공통 지지 제외율 %": round(100 * R["excluded_share"], 1),
    "탐지군 입금 약화 건수": cl.sup(R["obs"]), "대조군(표준화) 기대 건수": round(R["exp"], 1), "탐지군 약화율 %": round(100 * R["p_det"], 1),
    "표준화 대조군 약화율 %": round(100 * R["p_ctrl"], 1), "RR": round(R["rr"], 2), "95% CI": f"{R['lo']:.2f} ~ {R['hi']:.2f}", "단측 p": round(R["p_one"], 4),
    "채택 (CI 하한 > 1)": "채택" if R["adopt"] else "미채택", "유효 반복": R["B_valid"]}])
cl.show_table(tab.T.reset_index().rename(columns={"index": "항목", 0: "값"}), "H-C1 주 결과 (P36, v2)", OUT / "nb05", "hc1_main")
print(f"({time.time() - t0:.0f}초)  결론 문장(채택 시): '판정 가능하고 비교 가능한 기업에서 I1 최초 탐지가 향후 6개월의 상대적 입금 약화와 연관된다.'" if R["adopt"]
      else f"({time.time() - t0:.0f}초)  미채택: 연관의 증거를 얻지 못했다고 그대로 보고합니다 (n 재설정·재검정 없음).")
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(R["star"], bins=40, color="#2a78d6", alpha=0.85)
ax.axvline(1.0, color="k", ls="--", lw=1)
ax.axvline(R["rr"], color="#eb6834", lw=2)
ax.set_title("RR 부트스트랩 분포 (점선 = 1, 주황 = 점추정)")
plt.tight_layout(); plt.show()

## 5. ⑤-8 교란 점검: 균형표 · 입금 수준·추세 직접 조정(S1) · '입금 정상' 보조 · 기준월 제외
**탐지 전 입금 수준** L_pre = [t−11, t−6] 평균, **탐지 전 입금 추세** h_pre 는 **탐지 창 [t−5, t]와 겹치지 않는 구간**(k=6이면 t ≥ 24에서만 정의)입니다. 표준화 차이의 절댓값 **> 0.1** 이면 불균형으로 봅니다.

In [ ]:
cu, cg = vc.cells_main(U)
grade = dc.firm_attribute(panel, mk, "법인_고객등급")[U.firm]
duty = dc.firm_attribute(panel, mk, "전담고객여부")[U.firm]
sido = dc.firm_attribute(panel, mk, "사업장_시도")[U.firm]
lv_s = panel.attr_levels["사업장_시도"]
sido_all = dc.firm_attribute(panel, mk, "사업장_시도")
cnt_s = np.bincount(sido_all[sido_all >= 0], minlength=len(lv_s))
top_s = [int(x) for x in np.argsort(-cnt_s, kind="stable")[:5]]
covs = {}
for g_, nm in enumerate(SECN):
    covs[f"업종군: {nm}"] = (U.sector == g_).astype(float)
for q in range(5):
    covs[f"규모 {q + 1}분위"] = (U.size == q).astype(float)
for j, nm in enumerate(panel.attr_levels["법인_고객등급"]):
    covs[f"고객등급: {nm}"] = (grade == j).astype(float)
for j, nm in enumerate(panel.attr_levels["전담고객여부"]):
    covs[f"전담고객여부: {nm}"] = (duty == j).astype(float)
for j in top_s:
    covs[f"시도: {lv_s[j]}"] = (sido == j).astype(float)
covs["시도: 기타·결측"] = (~np.isin(sido, top_s)).astype(float)
covs["탐지 전 입금 수준 log(1+L_pre)"] = np.log1p(U.x["L_pre"])
covs["탐지 전 입금 추세 h_pre (e ≥ 23 만)"] = U.x["h_pre"]
rows = []
for nm, x in covs.items():
    r_ = vc.std_diff(x, U, cu, cg)
    rows.append({"변수": nm, "탐지군": round(r_["det"], 3), "대조군(조정 전)": round(r_["ctrl_raw"], 3), "대조군(표준화)": round(r_["ctrl_std"], 3),
                 "표준화 차이(조정 전)": round(r_["smd_raw"], 3), "표준화 차이(조정 후)": round(r_["smd_adj"], 3), "": "▲ >0.1" if abs(r_["smd_adj"]) > 0.1 else ""})
BAL = pd.DataFrame(rows)
cl.show_table(BAL, "균형표: 탐지군 vs 대조군 (표준화 차이)", OUT / "nb05", "balance")
print("조정 후에도 |표준화 차이| > 0.1 인 변수:", [r_["변수"] for r_ in rows if r_[""]] or "없음")

In [ ]:
# S1: 입금 수준·추세 직접 조정 (e ≥ 23).  보조: '입금 정상' 하위집단.  기준월 하나씩 제외.
mS1 = (U.e >= K + 17) & ~np.isnan(U.x["h_pre"]) & ~np.isnan(U.x["L_pre"])
Us1 = U.sub(mS1)
cs1 = vc.cells_s1(Us1, C5)
R_s1 = vc.analyze_units(Us1, F36, B, seed=20261011, cells=cs1)
Unorm = U.sub(mS1 & (U.x["h_pre"] > C5["theta_h"]))
R_norm = vc.analyze_units(Unorm, F36, B, seed=20261012)
R_s1m = vc.analyze_units(Us1, F36, B, seed=20261013)      # 같은 단위·주 층 (S1 과 비교용)
rows = [
    {"분석": "주 분석 (전체)", "탐지 기업": sup(R["n1"]), "RR (95% CI)": f_rr(R), "공통 지지 제외율 %": round(100 * R["excluded_share"], 1)},
    {"분석": "같은 단위(e ≥ 23), 주 층", "탐지 기업": sup(R_s1m["n1"]), "RR (95% CI)": f_rr(R_s1m), "공통 지지 제외율 %": round(100 * R_s1m["excluded_share"], 1)},
    {"분석": "S1: 입금 수준·추세 직접 조정 (e ≥ 23)", "탐지 기업": sup(R_s1["n1"]), "RR (95% CI)": f_rr(R_s1), "공통 지지 제외율 %": round(100 * R_s1["excluded_share"], 1)},
    {"분석": "보조: '입금 정상' 하위집단 (h_pre > θ_h) — 역인과를 제거하지 못함", "탐지 기업": sup(R_norm["n1"]), "RR (95% CI)": f_rr(R_norm), "공통 지지 제외율 %": round(100 * R_norm["excluded_share"], 1)}]
cl.show_table(pd.DataFrame(rows), "탐지 전 입금 수준·추세 조정 (S1)과 '입금 정상' 보조 분석", OUT / "nb05", "s1_and_normal")
rows = []
for e in ER:
    Ue = U.sub(U.e != e)
    cue, cge = vc.cells_main(Ue)
    est = vc.estimate(Ue, cue, cge)
    rows.append({"제외한 기준월": int(panel.months[e]), "탐지 기업": sup(est["n1"]), "RR": round(est["rr"], 2)})
LOO = pd.DataFrame(rows)
cl.show_table(LOO, "기준월 하나씩 제외한 RR (점추정)", OUT / "nb05", "leave_one_month")
print(f"RR 범위: {LOO['RR'].min():.2f} ~ {LOO['RR'].max():.2f} (전체 {R['rr']:.2f})")

## 6. ⑤-6 이질성 보고: 규모 5분위별 · 천장 유무별 (사전 지정, 주 결과를 바꾸지 않음)
탐지율이 규모에 따라 크게 다르므로 **층 안의 RR과 CI**를 따로 봅니다. 큰 기업의 탐지가 적어 CI가 넓으면 결론을 **"작은·중간 규모가 주도하는, 비교 가능한 층에서의 연관"** 으로 제한합니다.

In [ ]:
ceil_first = dc.ceiling_at_first_judgeable(panel, mk, ch, PHI2)
rows = []
def het_row(label, mask, firms_mask, cell_mask):
    Ug = U.sub(mask)
    Rg = vc.analyze_units(Ug, F36, B, seed=20261021 + len(rows))
    nd = int((det_any & firms_mask).sum())
    cq = jf & cell_mask
    rows.append({"집단": label, "판정 가능 기업": cl.sup(firms_mask.sum()), "탐지 기업": cl.sup(nd), "기업별 첫 탐지율 %": round(100 * nd / max(int(firms_mask.sum()), 1), 1),
                 "기업-월 문턱 통과율 %": round(100 * float((r_m[cq] <= -N2 / 100 + 1e-9).mean()), 1) if cq.any() else None,
                 "탐지군 약화율 %": round(100 * Rg["p_det"], 1) if np.isfinite(Rg["p_det"]) else None, "표준화 대조군 약화율 %": round(100 * Rg["p_ctrl"], 1) if np.isfinite(Rg["p_ctrl"]) else None,
                 "약화 사건(탐지군)": cl.sup(Rg["obs"]), "RR (95% CI)": f_rr(Rg), "공통 지지 제외율 %": round(100 * Rg["excluded_share"], 1) if np.isfinite(Rg["excluded_share"]) else None})
for q in range(5):
    het_row(f"규모 {q + 1}분위" + (" (최소)" if q == 0 else " (최대)" if q == 4 else ""), U.size == q, fsize == q, size_m == q)
FLm = D["FL"][:, ER]
for val, lab in ((True, "천장 보유 (첫 판정 가능월 비교 창에 50건초과 채널)"), (False, "천장 없음")):
    het_row(lab, ceil_first[U.firm] == val, (first >= 0) & (ceil_first == val), FLm == val)
HET = pd.DataFrame(rows)
cl.show_table(HET, "규모·천장 유무별 탐지와 RR (층 안 비교, P36)", OUT / "nb05", "heterogeneity")
wide = [r_["집단"] for r_ in rows if "(" in r_["RR (95% CI)"] and "정의" not in r_["RR (95% CI)"] and float(r_["RR (95% CI)"].split("~")[0].split("(")[1]) <= 1 <= float(r_["RR (95% CI)"].split("~")[1].rstrip(")"))]
undef = [r_["집단"] for r_ in rows if "정의 불가" in r_["RR (95% CI)"]]
print("CI 가 1 을 포함해 해당 집단의 연관을 확인할 수 없는 집단:", wide or "없음")
print("RR 정의 불가(대조군 약화 0건, 탐지 건수 부족 등) 집단:", undef or "없음")

## 7. ⑤-7 v2 수정의 점검 (Codex 4차): 제외된 칸의 특성 · v1 vs v2 · 천장 칸 제외(S-C1)
v2는 활동량 하한을 정할 때 **천장 채널이 있는 칸**을 뺐습니다. 이 제외가 방향 중립이라고 가정하지 않고 **그 칸의 규모·업종·변화 방향**을 봅니다.

In [ ]:
rows = []
sz_cells = D["size"][:, ER]
for flag, lab in ((True, "천장 채널이 있는 칸 (하한 산출에서 제외된 칸)"), (False, "천장 채널이 없는 칸")):
    m = (st[:, ER] == 0) & (FLm == flag)
    rr_ = ch.r[:, ER][m]
    row = {"집단": lab, "판정 가능 칸": cl.sup(m.sum()), "비중 %": round(100 * m.sum() / max(int(jf.sum()), 1), 1), "r < 0 비율 %": round(100 * float((rr_ < 0).mean()), 1),
           "r 중앙값 %": round(100 * float(np.median(rr_)), 1), "r ≤ −n 비율 %": round(100 * float((rr_ <= -N2 / 100 + 1e-9).mean()), 2)}
    for q in range(5):
        row[f"규모{q + 1} %"] = round(100 * float((sz_cells[m] == q).mean()), 1)
    rows.append(row)
cl.show_table(pd.DataFrame(rows), "하한 산출에서 제외된 칸(천장 칸)의 특성: 규모·변화 방향", OUT / "nb05", "v2_excluded_cells")
sec_cells = np.broadcast_to(D["sector"][:, None], (len(D["sector"]), len(ER)))
rows = []
for flag, lab in ((True, "천장 칸"), (False, "천장 없는 칸")):
    m = (st[:, ER] == 0) & (FLm == flag)
    rows.append({"집단": lab, **{nm: round(100 * float((sec_cells[m] == g_).mean()), 1) for g_, nm in enumerate(SECN)}})
cl.show_table(pd.DataFrame(rows), "같은 칸의 업종군 구성 (%)", OUT / "nb05", "v2_excluded_cells_sector")

In [ ]:
t0 = time.time()
D1 = vc.prepare(panel, mk, C5, phi=V1["phi"], n_pct=V1["n_pct"])
R_v1 = vc.analyze_units(D1["units"], F36, B, seed=20261031)
Dc1 = vc.prepare(panel, mk, C5, drop_ceiling_cells=True)
R_c1 = vc.analyze_units(Dc1["units"], F36, B, seed=20261032)
def jf_firms(Dx):
    return int((Dx["status"][:, ER] == 0).any(axis=1).sum())
rows = []
for lab, Dx, Rx in (("v2 (주 규칙, 결과 확인 전 개정안)", D, R), ("v1 (원래 등록 규칙, 민감도)", D1, R_v1), ("S-C1: 천장 채널이 있는 칸을 탐지 후보·위험집합에서 제외 (v2 규칙)", Dc1, R_c1)):
    jfx = jf_firms(Dx)
    nd = int((Dx["det"] >= 0).sum())
    rows.append({"규칙": lab, "Φ": round(Dx["phi"], 2), "n (%)": Dx["n_pct"], "판정 가능 기업": cl.sup(jfx), "판정 가능률 %": round(100 * jfx / F36, 1), "탐지 기업": cl.sup(nd),
                 "탐지율 % (판정 가능 중)": round(100 * nd / max(jfx, 1), 1), "RR (95% CI)": f_rr(Rx), "채택": "채택" if Rx["adopt"] else "미채택"})
V_TAB = pd.DataFrame(rows)
cl.show_table(V_TAB, "수정 전후 비교(v1 vs v2)와 S-C1 — 같은 θ_O1·규모 경계를 사용", OUT / "nb05", "v1_v2_sc1")
print(f"({time.time() - t0:.0f}초)  S-C2(순서 코드 합 하락 ΔI2 기반 탐지)는 P7 에서 같은 분석을 반복합니다.")

## 8. ⑤-9 H-C2: 입금 경로 기술 (탐색, 확증 판정 없음)
상대월 j (j = 0 이 기준월). **j ≤ −6 = 건수 감소 이전**, **j ∈ [−5, 0] = 탐지 창(건수가 줄던 시기, 동시 구간)**, j ∈ [1, 6] = 사후. 입금은 같은 달 전년 대비 **입금 합의 비**(평균의 변화율)를 층별로 구해 탐지군 층 가중으로 표준화하고 **Δ(j) = 탐지군 − 대조군**을 봅니다. 모든 j 가 정의되는 **완전 경로 이벤트(기준월 e ≥ 24)** 로 요약합니다.

In [ ]:
t0 = time.time()
J = list(range(-12, 7))
Up = U.sub(U.e >= 24)
cup, cgp = vc.cells_main(Up)
num, den, valid = vc.path_arrays(D["Yw"], Up.firm, Up.e, J)
n1_, n0_, _, _ = vc._counts(cup, int(cup.max()) + 1, np.ones(len(Up)), Up.is_det, np.zeros(len(Up)))
effp = vc.eff_cells(n1_, n0_, cgp)
delta, r1, r0, ns = vc.path_delta(num, den, valid, effp[cup], Up.is_det, np.ones(len(Up)))
sums, deltas = vc.bootstrap_path(Up, cup, cgp, num, den, valid, J, F36, B=B)
lo_d, hi_d = np.nanpercentile(deltas, 2.5, axis=0), np.nanpercentile(deltas, 97.5, axis=0)
Jn = np.array(J)
rows = [{"상대월 j": j, "구간": "건수 감소 이전" if j <= -6 else "탐지 창(동시)" if j <= 0 else "사후", "탐지군 이벤트(관측)": cl.sup(valid[Up.is_det, i].sum()),
         "탐지군 입금 전년비": round(r1[i], 3), "대조군(표준화) 전년비": round(r0[i], 3), "Δ(j)": round(delta[i], 3), "95% CI": f"{lo_d[i]:.3f} ~ {hi_d[i]:.3f}"} for i, j in enumerate(J)]
PATH = pd.DataFrame(rows)
cl.show_table(PATH, f"입금 경로 (완전 경로 이벤트 {int(Up.is_det.sum()):,}곳, P36)", OUT / "nb05", "hc2_path")
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axvspan(-5.5, 0.5, color="#eb6834", alpha=0.10)
ax.fill_between(Jn, lo_d, hi_d, color="#2a78d6", alpha=0.2)
ax.plot(Jn, delta, color="#2a78d6", marker="o")
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("상대월 j (음영 = 탐지 창, 건수가 줄던 시기)"); ax.set_ylabel("Δ = 탐지군 − 대조군 (입금 전년비)")
ax.set_title("입금 경로: 탐지군과 표준화 대조군의 차이")
plt.tight_layout(); plt.show()
cis = {k: vc.ci(v) for k, v in sums.items()}
pt = vc.path_summaries(delta, J)
tab = pd.DataFrame([{"요약": "건수 감소 이전 평균 Δ (j −12..−6)", "점추정": round(pt["pre_mean"], 3), "95% CI": f"{cis['pre_mean'][0]:.3f} ~ {cis['pre_mean'][1]:.3f}"},
                    {"요약": "사전 기울기 (j −12..−6 에서 Δ의 OLS 기울기)", "점추정": round(pt["slope_pre"], 4), "95% CI": f"{cis['slope_pre'][0]:.4f} ~ {cis['slope_pre'][1]:.4f}"},
                    {"요약": "동시 구간 평균 Δ (j −5..0)", "점추정": round(pt["conc_mean"], 3), "95% CI": f"{cis['conc_mean'][0]:.3f} ~ {cis['conc_mean'][1]:.3f}"},
                    {"요약": "사후 평균 Δ (j 1..6)", "점추정": round(pt["post_mean"], 3), "95% CI": f"{cis['post_mean'][0]:.3f} ~ {cis['post_mean'][1]:.3f}"}])
cl.show_table(tab, "H-C2 경로 요약 (판정 문턱 없음)", OUT / "nb05", "hc2_summary")
INTERP = vc.interpret_pre(cis["pre_mean"], cis["slope_pre"])
print("해석 문구:", INTERP, f" ({time.time() - t0:.0f}초)")

In [ ]:
# 모든 이벤트(e ≥ 17)로 그린 경로 (j ≥ −5 만 모든 이벤트에서 정의) + 민감도: 윈저화 없음 / 기업별 변화율의 중앙값
Ja = list(range(-5, 7))
numA, denA, valA = vc.path_arrays(D["Yw"], U.firm, U.e, Ja)
n1a, n0a, _, _ = vc._counts(cu, int(cu.max()) + 1, np.ones(len(U)), U.is_det, np.zeros(len(U)))
effA = vc.eff_cells(n1a, n0a, cg)
dA, r1A, r0A, nsA = vc.path_delta(numA, denA, valA, effA[cu], U.is_det, np.ones(len(U)))
numR, denR, valR = vc.path_arrays(D["Y"], Up.firm, Up.e, J)
dR, _, _, _ = vc.path_delta(numR, denR, valR, effp[cup], Up.is_det, np.ones(len(Up)))
dM = vc.path_median_delta(num, den, valid, effp[cup], Up.is_det)
rows = []
for i, j in enumerate(J):
    ia = Ja.index(j) if j in Ja else None
    rows.append({"상대월 j": j, "Δ 주 경로(윈저화, 합의 비)": round(delta[i], 3), "Δ 윈저화 없음": round(dR[i], 3), "Δ 기업별 변화율 중앙값": round(dM[i], 3),
                 "Δ 모든 이벤트(e ≥ 17)": round(dA[ia], 3) if ia is not None else None, "모든 이벤트 중 탐지군 관측": cl.sup(valA[U.is_det, ia].sum()) if ia is not None else None})
cl.show_table(pd.DataFrame(rows), "경로 민감도: 윈저화 없음 · 중앙값 · 모든 이벤트 (참고)", OUT / "nb05", "hc2_sensitivity")

## 9. 보조 결과와 참고표: O2 · O3 · n 격자 · 활동량 하한 미만 기업

In [ ]:
Uo3 = U.sub(U.x["o3_def"])
R_o3 = vc.analyze_units(Uo3, F36, B, seed=20261041, ev=Uo3.x["o3"])
o2 = np.minimum(U.x["o2"], C5["o2_cap"])
md_ = vc.mean_diff(U, cu, cg, o2)
star2 = vc.bootstrap_mean_diff(U, cu, cg, o2, F36, B=min(B, 1000))
lo2, hi2 = vc.ci(star2)
tab = pd.DataFrame([
    {"보조 결과": "O3: 잔액 약화 (판정 가능 전체 하위 5%)", "추정": f_rr(R_o3), "비고": "RR (95% CI)"},
    {"보조 결과": "O2: 입금 회복률 (e+4..e+6 / 전년 같은 달, 99% 상한 절단) 표준화 평균 차이", "추정": f"{md_['diff']:+.3f} ({lo2:+.3f}~{hi2:+.3f})", "비고": f"탐지군 {md_['det_mean']:.3f} vs 표준화 대조군 {md_['ctrl_std_mean']:.3f}"}])
cl.show_table(tab, "보조 결과 (O2·O3)", OUT / "nb05", "aux_o2_o3")
rows = []
for nn_ in range(20, 100, 10):
    dn = dc.first_detect(ch, PHI2, nn_ / 100.0)
    Un = vc.build_units(dn, st, D["G"], C5["theta_o1"], D["size"], D["sector"], ER)
    if int(Un.is_det.sum()) == 0:
        rows.append({"n (%)": nn_, "탐지 기업": 0})
        continue
    cun, cgn = vc.cells_main(Un)
    en = vc.estimate(Un, cun, cgn)
    rows.append({"n (%)": nn_, "탐지 기업": cl.sup(en["n1"]), "월평균 탐지 기업": round(en["n1"] / len(ER), 1), "탐지군 약화율 %": round(100 * en["p_det"], 1),
                 "표준화 대조군 약화율 %": round(100 * en["p_ctrl"], 1), "차이 %p": round(100 * (en["p_det"] - en["p_ctrl"]), 1), "RR (점추정)": round(en["rr"], 2)})
cl.show_table(pd.DataFrame(rows), "n 격자 참고표 (검정 없음 · 탐색: 규칙을 바꾸려면 잠긴 표본에서만 검증)", OUT / "nb05", "n_grid")
never = ~(st[:, ER] == 0).any(axis=1)
def_any = (~np.isnan(ch.r[:, ER])).any(axis=1)
G_er = D["G"][:, ER]
cells_low = (st[:, ER] == 2) & ~np.isnan(G_er)
cells_ok = (st[:, ER] == 0) & ~np.isnan(G_er)
sz23 = np.nanmean(D["I"][:, :12], axis=1)
rows = [{"집단": "하한 때문에 제외된 기업(한 번도 판정되지 못함)", "기업 수": cl.sup((never & def_any).sum()), "2023년 평균 I1 중앙값": round(float(np.nanmedian(sz23[never & def_any])), 1),
         "입금 약화 비율 % (하한 미만 칸)": round(100 * float((G_er[cells_low] <= C5["theta_o1"] + 1e-9).mean()), 1)},
        {"집단": "판정 가능 기업", "기업 수": cl.sup((~never).sum()), "2023년 평균 I1 중앙값": round(float(np.nanmedian(sz23[~never])), 1),
         "입금 약화 비율 % (판정 가능 칸)": round(100 * float((G_er[cells_ok] <= C5["theta_o1"] + 1e-9).mean()), 1)}]
cl.show_table(pd.DataFrame(rows), "활동량 하한으로 제외된 기업 (결론의 적용 범위는 '하한 이상 기업')", OUT / "nb05", "floor_excluded")

## 9-B. 사후 탐색 점검 (결과 확인 후 추가 · 규칙과 채택 판정은 바꾸지 않음)
H-C1 결과를 본 뒤 두 가지가 궁금해져 추가한 **탐색적 점검**입니다. 계획서에 미리 지정한 분석이 아니므로 P5 판정 기록에 '결과 확인 후 추가'로 따로 적습니다.
1. **방향 특이성**: 탐지군은 입금이 '약해지는' 꼬리만 더 많은가, 아니면 '크게 늘어나는' 꼬리도 더 많은가? 둘 다 많으면 '입금이 크게 흔들리는 기업'이 탐지에 섞여 있다는 뜻입니다.
2. **기저효과**: 입금 약화(O1)의 기준(전년 같은 달)은 탐지의 비교 기준 구간 [e−11, e−6]과 같은 달입니다. 이 구간에 입금이 **이미 급등해 있던** 기업은 건수와 입금이 함께 평소 수준으로 돌아오기만 해도 '입금 약화'로 잡힙니다. **급등이 없던 기업만** 비교해도 연관이 남는지 봅니다.

In [ ]:
# 사후 탐색 (결과 확인 후 추가): 고정 규칙·상수·채택 판정은 그대로 두고 보조 결과만 본다.
t0 = time.time()
# (1) 방향 특이성: 입금이 크게 '늘어난' 꼬리 (P36 판정 가능 전체 g 의 95번째 백분위수 이상)
theta_up = vc.theta_low(D["G"], st, ER, q=95.0)
ev_up = D["G"][U.firm, U.e] >= theta_up - 1e-9
R_up = vc.analyze_units(U, F36, B, seed=20261081, ev=ev_up)
def tail_row(lab, Rx):
    return {"결과": lab, "탐지 기업": sup(Rx["n1"]), "탐지군 비율 %": round(100 * Rx["p_det"], 1), "표준화 대조군 비율 %": round(100 * Rx["p_ctrl"], 1),
            "탐지군 사건 수": cl.sup(Rx["obs"]), "RR (95% CI)": f_rr(Rx)}
UPDOWN = pd.DataFrame([tail_row(f"입금 약화 꼬리 (g ≤ {C5['theta_o1']:.2f}, 주 결과 O1)", R), tail_row(f"입금 급증 꼬리 (g ≥ {theta_up:.2f}, 대칭 점검)", R_up)])
cl.show_table(UPDOWN, "방향 특이성: 입금 약화 꼬리와 급증 꼬리 (P36, 사후 탐색)", OUT / "nb05", "explore_updown")
if not np.isfinite(R_up["rr"]):
    print("→ 급증 꼬리 RR 정의 불가 (대조군 급증 사건이 없음)")
elif R_up["lo"] > 1:
    print("→ 급증 꼬리도 탐지군에서 더 많음: 입금이 약해지는 쪽만이 아니라 '입금이 크게 흔들리는 기업'이 탐지에 섞여 있음 (해석 가설)")
elif R_up["hi"] < 1:
    print("→ 급증 꼬리는 오히려 적음: 하락 방향에 특이적인 연관")
else:
    print("→ 급증 꼬리는 대조군과 구별되지 않음(CI가 1을 포함): 양방향 변동만으로는 설명되지 않음")

# (2) 기저효과: 탐지 전 입금 추세 h_pre 가 판정 가능 전체의 상위 10% 이상인 '급등' 기업을 나눠 본다 (탐지군·대조군에 같은 기준)
e_h = [e for e in ER if e >= K + 17]
hv = np.concatenate([D["H"][:, e][(st[:, e] == 0) & ~np.isnan(D["H"][:, e]) & ~np.isnan(D["L"][:, e])] for e in e_h])
h_p90 = float(np.percentile(hv, 90))
mB = (U.e >= K + 17) & ~np.isnan(U.x["h_pre"]) & ~np.isnan(U.x["L_pre"])
hp = U.x["h_pre"]
n_det_B, n_ctl_B = max(int((mB & U.is_det).sum()), 1), max(int((mB & ~U.is_det).sum()), 1)
groups = ((f"전체 (e ≥ {K + 17})", mB), (f"급등 없음 (h_pre ≤ {h_p90:.2f}, 판정 가능 전체의 상위 10% 경계 이하)", mB & (hp <= h_p90)), ("급등 (h_pre > 같은 경계)", mB & (hp > h_p90)))
rows, Rs = [], []
for i_, (lab, m_) in enumerate(groups):
    Ug = U.sub(m_)
    Rg = vc.analyze_units(Ug, F36, B, seed=20261090 + i_)
    Rs.append(Rg)
    nd_g, nc_g = int(Ug.is_det.sum()), int((~Ug.is_det).sum())
    rows.append({"집단": lab, "탐지 기업": cl.sup(nd_g), "탐지군 중 비중 %": round(100 * nd_g / n_det_B, 1), "위험집합 기업-월 중 비중 %": round(100 * nc_g / n_ctl_B, 1),
                 "탐지군 약화율 %": round(100 * Rg["p_det"], 1) if np.isfinite(Rg["p_det"]) else None,
                 "표준화 대조군 약화율 %": round(100 * Rg["p_ctrl"], 1) if np.isfinite(Rg["p_ctrl"]) else None,
                 "약화 사건(탐지군)": cl.sup(Rg["obs"]), "RR (95% CI)": f_rr(Rg), "공통 지지 제외율 %": round(100 * Rg["excluded_share"], 1) if np.isfinite(Rg["excluded_share"]) else None})
SURGE = pd.DataFrame(rows)
cl.show_table(SURGE, "탐지 직전 입금 급등 여부별 RR (P36, 사후 탐색)", OUT / "nb05", "explore_surge")
Rn = Rs[1]
if not np.isfinite(Rn["rr"]):
    print("→ 급등 없는 집단의 RR 정의 불가")
elif Rn["lo"] > 1:
    print("→ 급등이 없던 기업만 비교해도 연관이 남음: 탐지 직전 입금 급등 뒤 정상화(기저효과)만으로는 설명되지 않음 (해석 가설)")
else:
    print("→ 급등이 없던 기업에서는 연관의 증거가 확인되지 않음(CI가 1을 포함): 기저효과·평균 회귀의 영향이 클 수 있음 (표본이 줄어 검정력은 낮음)")
def _pk(r):
    return {k: (None if (isinstance(r[k], float) and not np.isfinite(r[k])) else float(r[k])) for k in ("rr", "lo", "hi", "p_det", "p_ctrl", "n1", "obs", "exp")}
cl.save_json(WORK / "interim" / "nb05_explore.json", {"theta_up": float(theta_up), "h_p90": h_p90, "up_tail": _pk(R_up),
             "surge_groups": {"전체": _pk(Rs[0]), "급등 없음": _pk(Rs[1]), "급등": _pk(Rs[2])}, "synthetic": bool(panel.synthetic)})
print(f"({time.time() - t0:.0f}초)")

## 10. 🔒 35개월 법인(P35만 · P35+): 잠금
**P36의 결과를 기록·해석하고 탐색을 마친 뒤**, 안내에 따라 `HOLDOUT_OPEN` 을 열어 이 셀을 실행합니다. P36에서 고정한 규칙·상수(θ_O1, 규모 경계, 업종군, Φ, n)를 **그대로** 적용합니다. 재현은 **P35만과 P36의 로그 비율비 차이와 그 CI**로 판단합니다 (두 표본은 겹치지 않는 기업이라 독립). P35+는 P36을 포함하므로 독립 증거가 아니며 참고용입니다.

In [ ]:
HOLD = {}
if not HOLDOUT_OPEN:
    print("🔒 잠김: 35개월 법인(P35만·P35+)의 입금 결과는 아직 열지 않았습니다. P36 결과 기록·해석 후 안내에 따라 엽니다.")
else:
    t0 = time.time()
    HOLD = {}
    for name, key, sd in (("P35만", "P35", 20261051), ("P35+", "P35p", 20261052)):
        Dh = vc.prepare(panel, masks[key], C5)
        Rh = vc.analyze_units(Dh["units"], Dh["F"], B, seed=sd)
        Dh1 = vc.prepare(panel, masks[key], C5, phi=V1["phi"], n_pct=V1["n_pct"])
        Rh1 = vc.analyze_units(Dh1["units"], Dh1["F"], B, seed=sd + 100)
        HOLD[name] = {"D": Dh, "R": Rh, "R_v1": Rh1}
    rows = []
    for name, Rx, Rv in (("P36 (결정 표본)", R, R_v1), ("P35만 (독립 재현)", HOLD["P35만"]["R"], HOLD["P35만"]["R_v1"]), ("P35+ (참고: P36 포함)", HOLD["P35+"]["R"], HOLD["P35+"]["R_v1"])):
        rows.append({"표본": name, "탐지 기업": cl.sup(Rx["n1"]), "RR v2 (95% CI)": f_rr(Rx), "채택 v2": "채택" if Rx["adopt"] else "미채택", "RR v1 (민감도)": f_rr(Rv),
                     "공통 지지 제외율 %": round(100 * Rx["excluded_share"], 1)})
    cl.show_table(pd.DataFrame(rows), "표본별 H-C1 (같은 규칙·같은 상수)", OUT / "nb05", "holdout_main")
    a, b_ = R["star"], HOLD["P35만"]["R"]["star"]
    m_ = min(len(a), len(b_))
    rng_ = np.random.default_rng(20261061)
    dlog = np.log(b_[rng_.permutation(len(b_))[:m_]]) - np.log(a[rng_.permutation(len(a))[:m_]])
    lo_, hi_ = np.percentile(dlog, [2.5, 97.5])
    print(f"재현 판단: log RR(P35만) − log RR(P36) = {np.log(HOLD['P35만']['R']['rr']) - np.log(R['rr']):+.3f} (95% CI {lo_:+.3f} ~ {hi_:+.3f}) → "
          + ("두 추정치의 차이가 0과 구별되지 않음" if lo_ <= 0 <= hi_ else "차이가 있음: 특성·공백 위치·판정 가능률을 비교해 '재현 실패'와 '표본 차이'를 구분 (P7)"))
    rows = []
    Dh = HOLD["P35만"]["D"]
    Uh = Dh["units"]
    for q in range(5):
        Rq = vc.analyze_units(Uh.sub(Uh.size == q), Dh["F"], B, seed=20261070 + q)
        rows.append({"집단": f"규모 {q + 1}분위", "탐지 기업": cl.sup(Rq["n1"]), "RR (95% CI)": f_rr(Rq)})
    cl.show_table(pd.DataFrame(rows), "P35만: 규모별 RR", OUT / "nb05", "holdout_size")
    print(f"({time.time() - t0:.0f}초)")

## 11. 요약 저장

In [ ]:
def pack(r):
    return {k: (None if (isinstance(v, float) and not np.isfinite(v)) else (bool(v) if isinstance(v, (bool, np.bool_)) else (float(v) if isinstance(v, (int, float, np.floating)) else v)))
            for k, v in r.items() if k != "star"}
summary = {"holdout_open": bool(HOLDOUT_OPEN), "rule": {"mode": MODE, "k": K, "phi": PHI2, "n_pct": N2},
           "constants": {"theta_o1": C5["theta_o1"], "theta_o3": C5["theta_o3"], "edges": [float(x) for x in C5["edges"]], "theta_h": C5["theta_h"]},
           "P36": {"main": pack(R), "v1": pack(R_v1), "sc1": pack(R_c1), "s1": pack(R_s1), "normal_subgroup": pack(R_norm), "o3": pack(R_o3),
                   "path": {"pre_mean": pt["pre_mean"], "slope_pre": pt["slope_pre"], "conc_mean": pt["conc_mean"], "post_mean": pt["post_mean"],
                            "pre_ci": list(cis["pre_mean"]), "slope_ci": list(cis["slope_pre"]), "interpretation": INTERP},
                   "n_stability": {"n0_2024": a0, "n0_2025": b0}},
           "synthetic": bool(panel.synthetic)}
if HOLD:
    summary["holdout"] = {k: {"main": pack(v["R"]), "v1": pack(v["R_v1"])} for k, v in HOLD.items()}
cl.save_json(WORK / "interim" / "nb05_summary.json", summary)
print(json.dumps({k: summary[k] for k in ("holdout_open", "rule")}, ensure_ascii=False))
print("P36 H-C1:", f_rr(R), "채택" if R["adopt"] else "미채택", "| 경로 해석:", INTERP)

## 체크포인트 (Claude에게 알려줄 것)
- [ ] **§1** 고정 상수: θ_O1, 규모 경계, 04와의 탐지 기업 수 일치
- [ ] **§2** n₀의 2024/2025 차이 (안정 여부)
- [ ] **§3** 규모별 판정 가능·탐지·**기업-월 문턱 통과율과 기업별 첫 탐지율**
- [ ] **§4** **H-C1: RR, 95% CI, 단측 p, 채택/미채택**, 탐지군·표준화 대조군 약화율, 공통 지지 제외율
- [ ] **§5** 균형표에서 조정 후에도 |표준화 차이| > 0.1 인 변수, **S1(입금 수준·추세 조정) RR**, '입금 정상' 보조 RR, 기준월 제외 RR 범위
- [ ] **§6** 규모 5분위별·천장 유무별 RR과 CI (CI 가 1 을 포함하는 집단)
- [ ] **§7** 제외된 칸(천장 칸)의 규모·변화 방향, **v1 vs v2 vs S-C1 의 RR**
- [ ] **§8** H-C2: 건수 감소 이전 평균 Δ·사전 기울기와 CI, 해석 문구, 동시·사후 평균 Δ
- [ ] **§9** O3·O2, n 격자 참고표, 하한 제외 기업
- [ ] **§9-B** (사후 탐색) 급증 꼬리 RR(방향 특이성), 급등 없음·급등 집단별 RR과 탐지군 중 급등 비중
- [ ] **§10** 🔒 잠금 상태 확인 (열지 않았음)

**다음**: P36 결과 판정 기록 → 탐색 마감 → 안내에 따라 §10 잠금 해제.